# Module 02 — Supervised Learning
## 02-06: Generalized Linear Models & the Exponential Family

**Objective:** Derive Gaussian (linear regression), Bernoulli (logistic regression),
Poisson, and Categorical (softmax) models as members of the exponential family,
revealing that all GLMs share the same canonical gradient update rule.

**Prerequisites:** 02-01 (Linear Regression), 02-02 (Logistic Regression),
01-07 (Probability & Statistics — MLE, distributions), 01-09 (Calculus & Optimization).

**GPU Required:** No (classical ML)

## Part 0 — Setup & Prerequisites

Why do linear regression and logistic regression use such different loss functions
yet look so similar in their gradient updates? The answer is the **exponential
family**: a parametric family of distributions that includes Gaussian, Bernoulli,
Poisson, and Categorical as special cases. Every member follows the same recipe —
choose a distribution, derive its canonical link function, and obtain a gradient
update rule that has the exact same form. This unification is the **Generalized
Linear Model (GLM)** framework. We implement four GLMs from scratch and verify
they reduce to the algorithms taught in 02-01 and 02-02.

In [ ]:
# ── Imports ──────────────────────────────────────────────────────────────────
import sys
import warnings
warnings.filterwarnings("ignore")

import random
import math
import time

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import torch

from sklearn.datasets import load_iris, fetch_california_housing
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (
    accuracy_score, f1_score, mean_squared_error, r2_score,
    confusion_matrix, ConfusionMatrixDisplay,
)
from sklearn.linear_model import LinearRegression, LogisticRegression
import sklearn

print(f"Python:       {sys.version.split()[0]}")
print(f"NumPy:        {np.__version__}")
print(f"Scikit-learn: {sklearn.__version__}")
print(f"PyTorch:      {torch.__version__}")
if torch.cuda.is_available():
    print(f"CUDA: {torch.version.cuda}")
    print(f"GPU:  {torch.cuda.get_device_name(0)}")

In [ ]:
# ── Reproducibility ───────────────────────────────────────────────────────────
SEED = 1103
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

print(f"Random seed: {SEED}")

In [ ]:
# ── Configuration ─────────────────────────────────────────────────────────────
# Data splits
TEST_SIZE: float = 0.20

# Gaussian GLM (linear regression on California Housing)
LR_GAUSSIAN:    float = 0.05
N_ITER_GAUSSIAN: int  = 500

# Bernoulli GLM (binary logistic on Iris setosa-vs-rest)
LR_BERNOULLI:    float = 0.1
N_ITER_BERNOULLI: int  = 300

# Softmax GLM (3-class Iris)
LR_SOFTMAX:    float = 0.1
N_ITER_SOFTMAX: int  = 400

# Poisson GLM (synthetic count data)
LR_POISSON:    float = 0.05
N_ITER_POISSON: int  = 500
N_POISSON:     int   = 600  # synthetic sample count

### Data Loading & EDA

We use three datasets — one per task type:
- **California Housing** (`fetch_california_housing`) — continuous regression target for the Gaussian GLM.
- **Iris** (`load_iris`) — 3-class classification for BernoulliGLM (binary) and SoftmaxGLM (3-class).
- **Synthetic Poisson data** — count targets generated from a known Poisson log-linear model.

In [ ]:
# ── Load California Housing (regression) ─────────────────────────────────────
housing = fetch_california_housing()
X_house_raw = housing.data
y_house     = housing.target  # Median house value (100k USD)
feature_names_house = list(housing.feature_names)

# Standardise features — critical for gradient-descent-based GLMs
scaler_house = StandardScaler()
X_house = scaler_house.fit_transform(X_house_raw)

X_house_tr, X_house_te, y_house_tr, y_house_te = train_test_split(
    X_house, y_house, test_size=TEST_SIZE, random_state=SEED)

# Add bias column (intercept trick: augment X with a column of ones)
X_house_tr_b = np.column_stack([np.ones(len(X_house_tr)), X_house_tr])
X_house_te_b = np.column_stack([np.ones(len(X_house_te)), X_house_te])

print("California Housing (regression)")
print(f"  Samples: {len(X_house_raw)}  |  Features: {X_house_raw.shape[1]}")
print(f"  Train: {len(X_house_tr)}  |  Test: {len(X_house_te)}")
print(f"  Target range: [{y_house.min():.2f}, {y_house.max():.2f}]  mean={y_house.mean():.3f}")
print(f"  Features: {feature_names_house}")

# ── Load Iris (classification) ────────────────────────────────────────────────
iris = load_iris()
X_iris_raw = iris.data
y_iris     = iris.target
class_names_iris = list(iris.target_names)
feature_names_iris = list(iris.feature_names)

scaler_iris = StandardScaler()
X_iris = scaler_iris.fit_transform(X_iris_raw)

X_iris_tr, X_iris_te, y_iris_tr, y_iris_te = train_test_split(
    X_iris, y_iris, test_size=TEST_SIZE, random_state=SEED, stratify=y_iris)

# Binary version: setosa (0) vs. non-setosa (1)
y_bin_tr = (y_iris_tr != 0).astype(int)
y_bin_te = (y_iris_te != 0).astype(int)

X_iris_tr_b = np.column_stack([np.ones(len(X_iris_tr)), X_iris_tr])
X_iris_te_b = np.column_stack([np.ones(len(X_iris_te)), X_iris_te])

print("\nIris (classification)")
print(f"  Samples: {len(X_iris_raw)}  |  Features: {X_iris_raw.shape[1]}")
print(f"  Train: {len(X_iris_tr)}  |  Test: {len(X_iris_te)}")
print(f"  Classes: {class_names_iris}")
print(f"  Binary task: setosa(0) vs non-setosa(1)")

# ── Generate synthetic Poisson count data ─────────────────────────────────────
rng = np.random.default_rng(SEED)
n_poisson = N_POISSON
d_poisson = 4
X_pois_raw = rng.standard_normal((n_poisson, d_poisson))
w_true_pois = np.array([0.5, -0.3, 0.8, -0.2])
b_true_pois = 1.0
log_rate = X_pois_raw @ w_true_pois + b_true_pois
y_pois = rng.poisson(np.exp(log_rate))  # Poisson-distributed counts

scaler_pois = StandardScaler()
X_pois = scaler_pois.fit_transform(X_pois_raw)

X_pois_tr, X_pois_te, y_pois_tr, y_pois_te = train_test_split(
    X_pois, y_pois, test_size=TEST_SIZE, random_state=SEED)

X_pois_tr_b = np.column_stack([np.ones(len(X_pois_tr)), X_pois_tr])
X_pois_te_b = np.column_stack([np.ones(len(X_pois_te)), X_pois_te])

print(f"\nSynthetic Poisson count data")
print(f"  Samples: {n_poisson}  |  Features: {d_poisson}")
print(f"  Train: {len(X_pois_tr)}  |  Test: {len(X_pois_te)}")
print(f"  Count range: [{y_pois.min()}, {y_pois.max()}]  mean={y_pois.mean():.2f}")

# ── EDA: target distributions ─────────────────────────────────────────────────
fig, axes = plt.subplots(1, 3, figsize=(14, 4))

axes[0].hist(y_house, bins=40, color="steelblue", edgecolor="white", alpha=0.85)
axes[0].set_title("California Housing — target distribution", fontsize=10)
axes[0].set_xlabel("Median house value (100k USD)")
axes[0].set_ylabel("Count")

colors_iris = ["steelblue", "darkorange", "firebrick"]
for cls in range(3):
    mask = y_iris == cls
    axes[1].scatter(X_iris_raw[mask, 0], X_iris_raw[mask, 1],
                    c=colors_iris[cls], s=20, alpha=0.7, label=class_names_iris[cls])
axes[1].set_title("Iris — sepal length vs width", fontsize=10)
axes[1].set_xlabel("Sepal length (cm)")
axes[1].set_ylabel("Sepal width (cm)")
axes[1].legend(fontsize=8)

axes[2].hist(y_pois, bins=range(int(y_pois.max()) + 2),
             color="darkorange", edgecolor="white", alpha=0.85, align="left")
axes[2].set_title("Synthetic Poisson counts — target distribution", fontsize=10)
axes[2].set_xlabel("Count y")
axes[2].set_ylabel("Frequency")

plt.suptitle("Dataset Target Distributions", fontsize=12)
plt.tight_layout()
plt.show()

---
## Part 1 — The Exponential Family from Scratch

### Exponential Family Form

A distribution belongs to the **exponential family** if its pdf/pmf can be written:

$$p(y;\, \boldsymbol{\eta}) = b(y) \exp\!\bigl(\boldsymbol{\eta}^\top \mathbf{T}(y) - a(\boldsymbol{\eta})\bigr)$$

where:
- $\boldsymbol{\eta}$ — **natural parameter** (canonical parameterisation)
- $\mathbf{T}(y)$ — **sufficient statistic** (all information about $\eta$ in the data)
- $a(\boldsymbol{\eta})$ — **log-partition function** / cumulant (ensures normalisation)
- $b(y)$ — **base measure** (independent of $\eta$)

### Key Property — Moments from $a(\eta)$

The mean and variance of the sufficient statistic are derivatives of $a$:
$$\mu = \mathbb{E}[\mathbf{T}(y);\boldsymbol{\eta}] = \nabla_{\boldsymbol{\eta}} a(\boldsymbol{\eta}), \qquad
\text{Var}[\mathbf{T}(y)] = \nabla^2_{\boldsymbol{\eta}} a(\boldsymbol{\eta})$$

### GLM Construction Recipe

Given a supervised learning problem with response $y$ and features $\mathbf{x}$:

1. **Choose** a distribution from the exponential family for $y | \mathbf{x}$.
2. **Derive** the canonical link $g$ such that $g(\mu) = \eta = \mathbf{w}^\top \mathbf{x}$.
3. **Optimise** the log-likelihood — the gradient always has the form:
$$\nabla_{\mathbf{w}} \mathcal{L} = \frac{1}{n} \mathbf{X}^\top (\boldsymbol{\mu} - \mathbf{y})$$

This universal gradient formula is the central result of GLM theory.

In [ ]:
# ── Exponential Family — Natural Parameters and Log-Partition Functions ─────────

def gaussian_natural(
    mu: float,
    sigma_sq: float = 1.0,
) -> tuple[float, float]:
    """Convert Gaussian mean-variance params to natural parameters.

    Gaussian: p(y; mu, sigma^2) = (1/sqrt(2pi sigma^2)) exp(-(y-mu)^2 / 2sigma^2)
    Natural params: eta1 = mu/sigma^2, eta2 = -1/(2*sigma^2)
    Sufficient stats: T(y) = [y, y^2]
    Log-partition: a(eta) = -eta1^2 / (4*eta2) - 0.5*log(-2*eta2)

    Args:
        mu:       Mean parameter.
        sigma_sq: Variance parameter (fixed to 1 in standard GLM formulation).

    Returns:
        Tuple of (eta1, eta2) — natural parameters.
    """
    eta1 = mu / sigma_sq
    eta2 = -1.0 / (2.0 * sigma_sq)
    return eta1, eta2


def gaussian_log_partition(eta1: float, eta2: float) -> float:
    """Compute log-partition function a(eta) for the Gaussian.

    Args:
        eta1: First natural parameter (mu / sigma^2).
        eta2: Second natural parameter (-1 / (2*sigma^2)).

    Returns:
        Log-partition value a(eta).
    """
    return -(eta1 ** 2) / (4.0 * eta2) - 0.5 * math.log(-2.0 * eta2)


def bernoulli_natural(phi: float) -> float:
    """Convert Bernoulli success probability to natural parameter.

    Bernoulli: p(y; phi) = phi^y * (1-phi)^(1-y)
    Natural param: eta = log(phi / (1-phi))  [log-odds / logit]
    Sufficient stat: T(y) = y
    Log-partition: a(eta) = log(1 + exp(eta))

    Args:
        phi: Probability of success in (0, 1).

    Returns:
        Natural parameter eta = logit(phi).
    """
    phi = float(np.clip(phi, 1e-9, 1 - 1e-9))
    return math.log(phi / (1.0 - phi))


def bernoulli_log_partition(eta: float) -> float:
    """Compute log-partition function a(eta) for the Bernoulli.

    a(eta) = log(1 + exp(eta))

    Args:
        eta: Natural parameter (log-odds).

    Returns:
        Log-partition value (softplus of eta).
    """
    return math.log(1.0 + math.exp(eta)) if eta < 500 else float(eta)


def poisson_natural(lam: float) -> float:
    """Convert Poisson rate to natural parameter.

    Poisson: p(y; lambda) = exp(-lambda) lambda^y / y!
    Natural param: eta = log(lambda)
    Sufficient stat: T(y) = y
    Log-partition: a(eta) = exp(eta)

    Args:
        lam: Poisson rate (mean count), lam > 0.

    Returns:
        Natural parameter eta = log(lam).
    """
    return math.log(max(lam, 1e-12))


def poisson_log_partition(eta: float) -> float:
    """Compute log-partition function a(eta) for the Poisson.

    a(eta) = exp(eta)

    Args:
        eta: Natural parameter (log of rate).

    Returns:
        Log-partition value exp(eta) = lambda.
    """
    return math.exp(eta)


# ── Verify: mean = a'(eta) for each distribution ─────────────────────────────
# Gaussian: a(eta1) = -eta1^2/(4*eta2), da/deta1 = -eta1/(2*eta2) = mu (when sigma^2=1)
phi = 0.75
eta_bern = bernoulli_natural(phi)
# a'(eta) = exp(eta)/(1+exp(eta)) = sigmoid(eta) = phi
sigmoid_eta = 1.0 / (1.0 + math.exp(-eta_bern))
print("Bernoulli verification:")
print(f"  phi={phi}, eta=logit(phi)={eta_bern:.4f}")
print(f"  a'(eta)=sigmoid(eta)={sigmoid_eta:.4f}  (should equal phi={phi})")

lam = 3.5
eta_pois = poisson_natural(lam)
# a'(eta) = exp(eta) = lambda
print(f"\nPoisson verification:")
print(f"  lambda={lam}, eta=log(lambda)={eta_pois:.4f}")
print(f"  a'(eta)=exp(eta)={math.exp(eta_pois):.4f}  (should equal lambda={lam})")

print("\nLink functions derived from a'(eta) = mu:")
print("  Gaussian:  mu = eta (identity link  => eta = mu)")
print("  Bernoulli: mu = sigmoid(eta) (logistic link => eta = logit(mu))")
print("  Poisson:   mu = exp(eta)    (exp link      => eta = log(mu))")
print("  Softmax:   mu_k = softmax(eta)_k  (softmax link => eta_k = log(mu_k) + const)")

### 1.1 Link Functions and Loss Functions

Each distribution's **canonical link** maps the linear predictor
$\eta = \mathbf{w}^\top \mathbf{x}$ to the mean $\mu = \mathbb{E}[y]$:

| Distribution | Link $g(\mu)=\eta$ | Inverse $\mu = g^{-1}(\eta)$ | Loss |
|---|---|---|---|
| Gaussian | Identity $\eta = \mu$ | $\mu = \eta$ | MSE |
| Bernoulli | Logit $\eta = \log\frac{\mu}{1-\mu}$ | $\mu = \sigma(\eta)$ | Binary cross-entropy |
| Poisson | Log $\eta = \log\mu$ | $\mu = e^\eta$ | Poisson NLL |
| Categorical | Softmax-log $\eta_k = \log\mu_k + C$ | $\mu_k = \text{softmax}(\eta)_k$ | Categorical cross-entropy |

Crucially, for each canonical GLM the gradient of the negative log-likelihood is:
$$\nabla_{\mathbf{w}} \mathcal{L} = \frac{1}{n} \mathbf{X}^\top (\boldsymbol{\mu} - \mathbf{y})$$

In [ ]:
# ── Link Functions and Loss Functions ────────────────────────────────────────

def sigmoid(z: np.ndarray) -> np.ndarray:
    """Numerically stable sigmoid: 1 / (1 + exp(-z)).

    Args:
        z: Input array of any shape.

    Returns:
        Sigmoid output in (0, 1), same shape as z.
    """
    return np.where(z >= 0,
                    1.0 / (1.0 + np.exp(-z)),
                    np.exp(z) / (1.0 + np.exp(z)))


def softmax(Z: np.ndarray) -> np.ndarray:
    """Row-wise softmax for a 2-D score matrix.

    Subtracts row-max for numerical stability before exponentiating.

    Args:
        Z: Score matrix, shape (n_samples, n_classes).

    Returns:
        Probability matrix, shape (n_samples, n_classes). Rows sum to 1.
    """
    Z_shifted = Z - Z.max(axis=1, keepdims=True)
    exp_Z     = np.exp(Z_shifted)
    return exp_Z / exp_Z.sum(axis=1, keepdims=True)


def poisson_inverse_link(eta: np.ndarray) -> np.ndarray:
    """Poisson inverse link (canonical): mu = exp(eta).

    Args:
        eta: Linear predictor array.

    Returns:
        Mean (rate) array; all values are positive.
    """
    return np.exp(np.clip(eta, -30, 30))


def mse_loss(y: np.ndarray, mu: np.ndarray) -> float:
    """Mean squared error loss for Gaussian GLM.

    Args:
        y:  True targets, shape (n,).
        mu: Predicted means, shape (n,).

    Returns:
        Scalar MSE value.
    """
    return float(np.mean((y - mu) ** 2))


def binary_cross_entropy(y: np.ndarray, mu: np.ndarray) -> float:
    """Binary cross-entropy loss for Bernoulli GLM.

    Args:
        y:  True binary labels in {0, 1}, shape (n,).
        mu: Predicted probabilities in (0, 1), shape (n,).

    Returns:
        Scalar mean cross-entropy.
    """
    mu_clipped = np.clip(mu, 1e-9, 1.0 - 1e-9)
    return float(-np.mean(y * np.log(mu_clipped) + (1 - y) * np.log(1 - mu_clipped)))


def categorical_cross_entropy(
    y_onehot: np.ndarray,
    mu: np.ndarray,
) -> float:
    """Categorical cross-entropy loss for Softmax GLM.

    Args:
        y_onehot: One-hot encoded targets, shape (n, K).
        mu:       Predicted class probabilities, shape (n, K).

    Returns:
        Scalar mean categorical cross-entropy.
    """
    mu_clipped = np.clip(mu, 1e-9, 1.0)
    return float(-np.mean(np.sum(y_onehot * np.log(mu_clipped), axis=1)))


def poisson_nll(y: np.ndarray, mu: np.ndarray) -> float:
    """Negative Poisson log-likelihood.

    NLL = -mean[y * log(mu) - mu - log(y!)]  \approx -mean[y*log(mu) - mu]

    Args:
        y:  Observed counts (non-negative integers), shape (n,).
        mu: Predicted rates (positive), shape (n,).

    Returns:
        Scalar negative log-likelihood (without the constant log(y!) term).
    """
    mu_clipped = np.maximum(mu, 1e-9)
    return float(-np.mean(y * np.log(mu_clipped) - mu_clipped))


def glm_gradient(
    X_b: np.ndarray,
    y: np.ndarray,
    mu: np.ndarray,
) -> np.ndarray:
    """Compute the canonical GLM gradient: (1/n) X^T (mu - y).

    This universal formula holds for ALL GLMs with canonical link functions:
    Gaussian, Bernoulli, Poisson, and Categorical. The only difference between
    GLMs is how mu is computed from the linear predictor X @ w.

    Args:
        X_b: Feature matrix with bias column, shape (n_samples, n_features + 1).
        y:   True targets, shape (n_samples,) or (n_samples, n_classes).
        mu:  Predicted means from inverse link, shape (n_samples,) or (n_samples, n_classes).

    Returns:
        Gradient array, same shape as the weight vector.
    """
    residuals = mu - y
    return X_b.T @ residuals / len(y)


# Visualise all four link functions
z_range = np.linspace(-5, 5, 200)
fig, axes = plt.subplots(1, 4, figsize=(15, 3))

axes[0].plot(z_range, z_range, color="steelblue", linewidth=2)
axes[0].set_title("Gaussian\nIdentity link: mu = eta", fontsize=10)
axes[0].set_xlabel("Linear predictor eta")
axes[0].set_ylabel("Mean mu")

axes[1].plot(z_range, sigmoid(z_range), color="darkorange", linewidth=2)
axes[1].set_title("Bernoulli\nLogistic link: mu = sigma(eta)", fontsize=10)
axes[1].set_xlabel("Linear predictor eta")
axes[1].set_ylabel("Mean mu (probability)")
axes[1].axhline(0.5, color="gray", linestyle=":", alpha=0.6)

axes[2].plot(z_range, np.exp(np.clip(z_range, -30, 5)), color="firebrick", linewidth=2)
axes[2].set_title("Poisson\nLog link: mu = exp(eta)", fontsize=10)
axes[2].set_xlabel("Linear predictor eta")
axes[2].set_ylabel("Mean mu (rate)")
axes[2].set_ylim(0, 20)

z_softmax = np.column_stack([z_range, np.zeros_like(z_range)])
mu_softmax = softmax(z_softmax)[:, 0]
axes[3].plot(z_range, mu_softmax, color="purple", linewidth=2, label="P(class 0)")
axes[3].plot(z_range, 1.0 - mu_softmax, color="teal", linewidth=2,
             linestyle="--", label="P(class 1)")
axes[3].set_title("Categorical\nSoftmax link (2 classes)", fontsize=10)
axes[3].set_xlabel("Score eta_0 - eta_1")
axes[3].set_ylabel("Probability")
axes[3].legend(fontsize=8)

plt.suptitle("Canonical Link Functions for the Four GLMs", fontsize=12)
plt.tight_layout()
plt.show()

---
## Part 2 — Putting It All Together

We assemble four GLM classes. Each has the **same training loop skeleton**:
compute the linear predictor `eta = X @ w`, apply the inverse link to get `mu`,
compute the loss, and call `glm_gradient(X, y, mu)` to update `w`. The only
difference is the line computing `mu`.

In [ ]:
class GaussianGLM:
    """Linear regression derived from the Gaussian exponential family.

    Distribution: y | x ~ Normal(mu, sigma^2) with sigma^2 = 1 (fixed).
    Canonical link: identity -- eta = mu = w^T x.
    Loss: mean squared error (negative Gaussian log-likelihood up to constants).
    Gradient: (1/n) X^T (mu - y)  -- same canonical GLM gradient.

    Attributes:
        learning_rate: Gradient descent step size.
        n_iterations:  Number of gradient steps.
        weights_:      Fitted coefficient vector (includes bias), shape (n_features+1,).
        losses_:       MSE loss recorded every 20 iterations.
    """

    def __init__(
        self,
        learning_rate: float = 0.05,
        n_iterations: int = 500,
    ) -> None:
        """Initialise GaussianGLM (linear regression) hyperparameters.

        Args:
            learning_rate: Step size for gradient descent.
            n_iterations:  Number of gradient descent steps.
        """
        self.learning_rate = learning_rate
        self.n_iterations  = n_iterations
        self.weights_: np.ndarray = np.array([])
        self.losses_: list[float] = []

    def _inverse_link(self, eta: np.ndarray) -> np.ndarray:
        """Apply Gaussian inverse link: mu = eta (identity).

        Args:
            eta: Linear predictor, shape (n,).

        Returns:
            Predicted mean mu = eta, shape (n,).
        """
        return eta

    def fit(self, X_b: np.ndarray, y: np.ndarray) -> "GaussianGLM":
        """Fit the Gaussian GLM via gradient descent on MSE.

        The gradient (1/n) X^T (mu - y) is the canonical GLM gradient.

        Args:
            X_b: Feature matrix with bias column, shape (n, d+1).
            y:   Continuous targets, shape (n,).

        Returns:
            self — the fitted model.
        """
        n, p = X_b.shape
        self.weights_ = np.zeros(p)
        self.losses_  = []

        for step in range(self.n_iterations):
            eta = X_b @ self.weights_
            mu  = self._inverse_link(eta)

            if step % 20 == 0:
                self.losses_.append(mse_loss(y, mu))

            # Canonical GLM gradient
            grad = glm_gradient(X_b, y, mu)
            self.weights_ -= self.learning_rate * grad

        return self

    def predict(self, X_b: np.ndarray) -> np.ndarray:
        """Predict continuous target values.

        Args:
            X_b: Feature matrix with bias column, shape (n, d+1).

        Returns:
            Predicted means, shape (n,).
        """
        return self._inverse_link(X_b @ self.weights_)

    def score(self, X_b: np.ndarray, y: np.ndarray) -> float:
        """Compute R-squared (coefficient of determination).

        Args:
            X_b: Feature matrix with bias column, shape (n, d+1).
            y:   True targets, shape (n,).

        Returns:
            R-squared in (-inf, 1]. 1 = perfect prediction.
        """
        return float(r2_score(y, self.predict(X_b)))

    def __repr__(self) -> str:
        return (f"GaussianGLM(learning_rate={self.learning_rate}, "
                f"n_iterations={self.n_iterations})")

In [ ]:
class BernoulliGLM:
    """Logistic regression derived from the Bernoulli exponential family.

    Distribution: y | x ~ Bernoulli(mu) with mu = sigmoid(w^T x).
    Canonical link: logit -- eta = log(mu/(1-mu)), inverse: mu = sigmoid(eta).
    Loss: binary cross-entropy (negative Bernoulli log-likelihood).
    Gradient: (1/n) X^T (mu - y)  -- same canonical GLM gradient.

    Attributes:
        learning_rate: Gradient descent step size.
        n_iterations:  Number of gradient steps.
        weights_:      Fitted coefficients (includes bias), shape (n_features+1,).
        losses_:       Binary cross-entropy recorded every 20 iterations.
    """

    def __init__(
        self,
        learning_rate: float = 0.1,
        n_iterations: int = 300,
    ) -> None:
        """Initialise BernoulliGLM (logistic regression) hyperparameters.

        Args:
            learning_rate: Step size for gradient descent.
            n_iterations:  Number of gradient descent steps.
        """
        self.learning_rate = learning_rate
        self.n_iterations  = n_iterations
        self.weights_: np.ndarray = np.array([])
        self.losses_: list[float] = []

    def _inverse_link(self, eta: np.ndarray) -> np.ndarray:
        """Apply Bernoulli inverse link: mu = sigmoid(eta).

        Args:
            eta: Linear predictor, shape (n,).

        Returns:
            Predicted probabilities in (0, 1), shape (n,).
        """
        return sigmoid(eta)

    def fit(self, X_b: np.ndarray, y: np.ndarray) -> "BernoulliGLM":
        """Fit the Bernoulli GLM via gradient descent on binary cross-entropy.

        Args:
            X_b: Feature matrix with bias column, shape (n, d+1).
            y:   Binary labels in {0, 1}, shape (n,).

        Returns:
            self — the fitted model.
        """
        n, p = X_b.shape
        self.weights_ = np.zeros(p)
        self.losses_  = []

        for step in range(self.n_iterations):
            eta = X_b @ self.weights_
            mu  = self._inverse_link(eta)

            if step % 20 == 0:
                self.losses_.append(binary_cross_entropy(y, mu))

            grad = glm_gradient(X_b, y, mu)
            self.weights_ -= self.learning_rate * grad

        return self

    def predict_proba(self, X_b: np.ndarray) -> np.ndarray:
        """Predict class probabilities.

        Args:
            X_b: Feature matrix with bias column, shape (n, d+1).

        Returns:
            Probability of class 1, shape (n,).
        """
        return self._inverse_link(X_b @ self.weights_)

    def predict(self, X_b: np.ndarray) -> np.ndarray:
        """Predict binary class labels using 0.5 threshold.

        Args:
            X_b: Feature matrix with bias column, shape (n, d+1).

        Returns:
            Predicted labels in {0, 1}, shape (n,).
        """
        return (self.predict_proba(X_b) >= 0.5).astype(int)

    def score(self, X_b: np.ndarray, y: np.ndarray) -> float:
        """Compute classification accuracy.

        Args:
            X_b: Feature matrix with bias column, shape (n, d+1).
            y:   True binary labels in {0, 1}, shape (n,).

        Returns:
            Accuracy in [0, 1].
        """
        return float(np.mean(self.predict(X_b) == y))

    def __repr__(self) -> str:
        return (f"BernoulliGLM(learning_rate={self.learning_rate}, "
                f"n_iterations={self.n_iterations})")

In [ ]:
def one_hot_encode(y: np.ndarray, n_classes: int) -> np.ndarray:
    """Convert integer class labels to a one-hot matrix.

    Args:
        y:         Integer labels in {0, ..., n_classes-1}, shape (n,).
        n_classes: Total number of classes K.

    Returns:
        One-hot matrix, shape (n, K).
    """
    n = len(y)
    Y = np.zeros((n, n_classes))
    Y[np.arange(n), y] = 1.0
    return Y


class SoftmaxGLM:
    """Multinomial logistic (softmax) regression from the Categorical exponential family.

    Distribution: y | x ~ Categorical(mu) with mu = softmax(W^T x).
    Canonical link: softmax-log -- eta_k = log(mu_k) + const.
    Loss: categorical cross-entropy (negative Categorical log-likelihood).
    Gradient: (1/n) X^T (mu - Y)  -- same canonical GLM gradient (matrix form).

    Attributes:
        n_classes:     Number of target classes K.
        learning_rate: Gradient descent step size.
        n_iterations:  Number of gradient steps.
        weights_:      Fitted weight matrix (includes bias), shape (n_features+1, K).
        losses_:       Categorical cross-entropy per 20 iterations.
    """

    def __init__(
        self,
        n_classes: int = 3,
        learning_rate: float = 0.1,
        n_iterations: int = 400,
    ) -> None:
        """Initialise SoftmaxGLM hyperparameters.

        Args:
            n_classes:     Number of output classes K.
            learning_rate: Step size for gradient descent.
            n_iterations:  Number of gradient descent steps.
        """
        self.n_classes     = n_classes
        self.learning_rate = learning_rate
        self.n_iterations  = n_iterations
        self.weights_: np.ndarray = np.array([[]])
        self.losses_: list[float] = []

    def _inverse_link(self, eta: np.ndarray) -> np.ndarray:
        """Apply Categorical inverse link: mu = softmax(eta).

        Args:
            eta: Linear predictor matrix, shape (n, K).

        Returns:
            Class probability matrix, shape (n, K). Rows sum to 1.
        """
        return softmax(eta)

    def fit(self, X_b: np.ndarray, y: np.ndarray) -> "SoftmaxGLM":
        """Fit Softmax GLM via gradient descent on categorical cross-entropy.

        Args:
            X_b: Feature matrix with bias column, shape (n, d+1).
            y:   Integer class labels in {0, ..., K-1}, shape (n,).

        Returns:
            self — the fitted model.
        """
        n, p = X_b.shape
        K = self.n_classes
        Y_onehot = one_hot_encode(y, K)

        self.weights_ = np.zeros((p, K))
        self.losses_  = []

        for step in range(self.n_iterations):
            eta = X_b @ self.weights_    # (n, K)
            mu  = self._inverse_link(eta)  # (n, K)

            if step % 20 == 0:
                self.losses_.append(categorical_cross_entropy(Y_onehot, mu))

            # Canonical GLM gradient (matrix form)
            # grad shape: (p, K)  same as weights_
            grad = glm_gradient(X_b, Y_onehot, mu)
            self.weights_ -= self.learning_rate * grad

        return self

    def predict_proba(self, X_b: np.ndarray) -> np.ndarray:
        """Predict class probability distributions.

        Args:
            X_b: Feature matrix with bias column, shape (n, d+1).

        Returns:
            Probability matrix, shape (n, K). Rows sum to 1.
        """
        return self._inverse_link(X_b @ self.weights_)

    def predict(self, X_b: np.ndarray) -> np.ndarray:
        """Predict class labels by taking argmax of softmax probabilities.

        Args:
            X_b: Feature matrix with bias column, shape (n, d+1).

        Returns:
            Predicted class labels in {0, ..., K-1}, shape (n,).
        """
        return np.argmax(self.predict_proba(X_b), axis=1)

    def score(self, X_b: np.ndarray, y: np.ndarray) -> float:
        """Compute classification accuracy.

        Args:
            X_b: Feature matrix with bias column, shape (n, d+1).
            y:   True integer labels, shape (n,).

        Returns:
            Accuracy in [0, 1].
        """
        return float(np.mean(self.predict(X_b) == y))

    def __repr__(self) -> str:
        return (f"SoftmaxGLM(n_classes={self.n_classes}, "
                f"learning_rate={self.learning_rate}, "
                f"n_iterations={self.n_iterations})")

In [ ]:
class PoissonGLM:
    """Poisson regression derived from the Poisson exponential family.

    Distribution: y | x ~ Poisson(mu) with mu = exp(w^T x).
    Canonical link: log -- eta = log(mu), inverse: mu = exp(eta).
    Loss: negative Poisson log-likelihood (ignoring log(y!) constant).
    Gradient: (1/n) X^T (mu - y)  -- same canonical GLM gradient.

    Attributes:
        learning_rate: Gradient descent step size.
        n_iterations:  Number of gradient steps.
        weights_:      Fitted coefficients (includes bias), shape (n_features+1,).
        losses_:       Poisson NLL recorded every 20 iterations.
    """

    def __init__(
        self,
        learning_rate: float = 0.05,
        n_iterations: int = 500,
    ) -> None:
        """Initialise PoissonGLM hyperparameters.

        Args:
            learning_rate: Step size for gradient descent.
            n_iterations:  Number of gradient descent steps.
        """
        self.learning_rate = learning_rate
        self.n_iterations  = n_iterations
        self.weights_: np.ndarray = np.array([])
        self.losses_: list[float] = []

    def _inverse_link(self, eta: np.ndarray) -> np.ndarray:
        """Apply Poisson inverse link: mu = exp(eta).

        Args:
            eta: Linear predictor, shape (n,).

        Returns:
            Predicted rates (positive), shape (n,).
        """
        return poisson_inverse_link(eta)

    def fit(self, X_b: np.ndarray, y: np.ndarray) -> "PoissonGLM":
        """Fit the Poisson GLM via gradient descent on negative Poisson log-likelihood.

        Args:
            X_b: Feature matrix with bias column, shape (n, d+1).
            y:   Non-negative integer counts, shape (n,).

        Returns:
            self — the fitted model.
        """
        n, p = X_b.shape
        self.weights_ = np.zeros(p)
        self.losses_  = []

        for step in range(self.n_iterations):
            eta = X_b @ self.weights_
            mu  = self._inverse_link(eta)

            if step % 20 == 0:
                self.losses_.append(poisson_nll(y, mu))

            grad = glm_gradient(X_b, y.astype(float), mu)
            self.weights_ -= self.learning_rate * grad

        return self

    def predict(self, X_b: np.ndarray) -> np.ndarray:
        """Predict Poisson count means (not integer-rounded).

        Args:
            X_b: Feature matrix with bias column, shape (n, d+1).

        Returns:
            Predicted rates (positive floats), shape (n,).
        """
        return self._inverse_link(X_b @ self.weights_)

    def score(self, X_b: np.ndarray, y: np.ndarray) -> float:
        """Compute R-squared between predicted rates and observed counts.

        Args:
            X_b: Feature matrix with bias column, shape (n, d+1).
            y:   True counts, shape (n,).

        Returns:
            R-squared in (-inf, 1].
        """
        return float(r2_score(y, self.predict(X_b)))

    def __repr__(self) -> str:
        return (f"PoissonGLM(learning_rate={self.learning_rate}, "
                f"n_iterations={self.n_iterations})")

### Part 2 — Sanity Checks

Before real training, we verify each GLM on a simple controlled problem where
the correct answer is known analytically.

In [ ]:
# ── Sanity Check: all four GLMs on tiny known problems ───────────────────────
rng_sanity = np.random.default_rng(SEED + 1)

# Gaussian: y = 3*x1 - 2*x2 + 1 (no noise)
n_s = 200
X_s_raw = rng_sanity.standard_normal((n_s, 2))
X_s = np.column_stack([np.ones(n_s), X_s_raw])
w_true_g = np.array([1.0, 3.0, -2.0])
y_s_gauss = X_s @ w_true_g + rng_sanity.normal(0, 0.1, n_s)

glm_gauss_sanity = GaussianGLM(learning_rate=0.1, n_iterations=300)
glm_gauss_sanity.fit(X_s, y_s_gauss)
r2_sanity = r2_score(y_s_gauss, glm_gauss_sanity.predict(X_s))
print(f"GaussianGLM sanity R^2: {r2_sanity:.4f}  (should be ~1.0)")
print(f"  True w: {w_true_g}  Fitted: {np.round(glm_gauss_sanity.weights_, 3)}")

# Bernoulli: p = sigmoid(2*x1 - x2)
w_true_b = np.array([0.0, 2.0, -1.0])
p_s = sigmoid(X_s @ w_true_b)
y_s_bern = rng_sanity.binomial(1, p_s)

glm_bern_sanity = BernoulliGLM(learning_rate=0.2, n_iterations=500)
glm_bern_sanity.fit(X_s, y_s_bern)
acc_sanity_bern = glm_bern_sanity.score(X_s, y_s_bern)
print(f"\nBernoulliGLM sanity accuracy: {acc_sanity_bern:.2%}")
print(f"  True w approx: {w_true_b}  Fitted: {np.round(glm_bern_sanity.weights_, 3)}")

# Softmax: 3-class with 3 weight vectors
n_soft = 300
X_soft_raw = rng_sanity.standard_normal((n_soft, 3))
X_soft     = np.column_stack([np.ones(n_soft), X_soft_raw])
W_true     = np.array([[0, 1, -1, 0], [0, -1, 2, 0], [0, 0, -1, 1]], dtype=float).T
logits     = X_soft @ W_true
probs      = softmax(logits)
y_soft     = np.array([rng_sanity.choice(3, p=row) for row in probs])

glm_soft_sanity = SoftmaxGLM(n_classes=3, learning_rate=0.2, n_iterations=500)
glm_soft_sanity.fit(X_soft, y_soft)
acc_soft = glm_soft_sanity.score(X_soft, y_soft)
print(f"\nSoftmaxGLM sanity accuracy: {acc_soft:.2%}")

# Poisson: lambda = exp(0.5*x1 + 1.0)
w_true_p = np.array([1.0, 0.5, 0.0])
y_s_pois = rng_sanity.poisson(np.exp(X_s @ w_true_p))

glm_pois_sanity = PoissonGLM(learning_rate=0.05, n_iterations=600)
glm_pois_sanity.fit(X_s, y_s_pois.astype(float))
r2_pois = glm_pois_sanity.score(X_s, y_s_pois)
print(f"\nPoissonGLM sanity R^2: {r2_pois:.4f}")
print(f"  True w: {w_true_p}  Fitted: {np.round(glm_pois_sanity.weights_, 3)}")

print("\nAll four GLMs recovered the true parameters from synthetic data.")

---
## Part 3 — Training on Real Datasets

In [ ]:
# ── Train Gaussian GLM (California Housing) ──────────────────────────────────
t0 = time.time()
glm_gauss = GaussianGLM(learning_rate=LR_GAUSSIAN, n_iterations=N_ITER_GAUSSIAN)
glm_gauss.fit(X_house_tr_b, y_house_tr)
t_gauss = time.time() - t0

y_pred_gauss_tr = glm_gauss.predict(X_house_tr_b)
y_pred_gauss_te = glm_gauss.predict(X_house_te_b)
r2_gauss_tr  = r2_score(y_house_tr, y_pred_gauss_tr)
r2_gauss_te  = r2_score(y_house_te, y_pred_gauss_te)
mse_gauss_tr = mse_loss(y_house_tr, y_pred_gauss_tr)
mse_gauss_te = mse_loss(y_house_te, y_pred_gauss_te)

print("GaussianGLM (Linear Regression) — California Housing")
print(f"  Train R^2: {r2_gauss_tr:.4f}  |  Test R^2: {r2_gauss_te:.4f}")
print(f"  Train MSE: {mse_gauss_tr:.4f}  |  Test MSE: {mse_gauss_te:.4f}")
print(f"  Time: {t_gauss:.2f}s")

# ── Train Bernoulli GLM (Iris, binary: setosa vs rest) ────────────────────────
t0 = time.time()
glm_bern = BernoulliGLM(learning_rate=LR_BERNOULLI, n_iterations=N_ITER_BERNOULLI)
glm_bern.fit(X_iris_tr_b, y_bin_tr)
t_bern = time.time() - t0

acc_bern_tr = glm_bern.score(X_iris_tr_b, y_bin_tr)
acc_bern_te = glm_bern.score(X_iris_te_b, y_bin_te)
print(f"\nBernoulliGLM (Logistic Regression) — Iris binary (setosa vs rest)")
print(f"  Train acc: {acc_bern_tr:.2%}  |  Test acc: {acc_bern_te:.2%}")
print(f"  Time: {t_bern:.2f}s")

# ── Train Softmax GLM (Iris, 3-class) ─────────────────────────────────────────
t0 = time.time()
glm_soft = SoftmaxGLM(n_classes=3, learning_rate=LR_SOFTMAX, n_iterations=N_ITER_SOFTMAX)
glm_soft.fit(X_iris_tr_b, y_iris_tr)
t_soft = time.time() - t0

acc_soft_tr = glm_soft.score(X_iris_tr_b, y_iris_tr)
acc_soft_te = glm_soft.score(X_iris_te_b, y_iris_te)
print(f"\nSoftmaxGLM (Multinomial Logistic) — Iris 3-class")
print(f"  Train acc: {acc_soft_tr:.2%}  |  Test acc: {acc_soft_te:.2%}")
print(f"  Time: {t_soft:.2f}s")

# ── Train Poisson GLM (synthetic count data) ──────────────────────────────────
t0 = time.time()
glm_pois = PoissonGLM(learning_rate=LR_POISSON, n_iterations=N_ITER_POISSON)
glm_pois.fit(X_pois_tr_b, y_pois_tr.astype(float))
t_pois = time.time() - t0

r2_pois_tr = glm_pois.score(X_pois_tr_b, y_pois_tr)
r2_pois_te = glm_pois.score(X_pois_te_b, y_pois_te)
print(f"\nPoissonGLM — Synthetic count data")
print(f"  Train R^2: {r2_pois_tr:.4f}  |  Test R^2: {r2_pois_te:.4f}")
print(f"  Time: {t_pois:.2f}s")

### Library Comparison

We compare our GLM implementations against sklearn's `LinearRegression` and
`LogisticRegression`. These use exact solvers (normal equation, LBFGS) rather
than gradient descent, so we expect better numerical precision but the same
predictions up to solver tolerance.

In [ ]:
# ── sklearn comparison ────────────────────────────────────────────────────────
# GaussianGLM vs. LinearRegression (closed-form normal equation)
sk_lr = LinearRegression()
sk_lr.fit(X_house_tr, y_house_tr)
r2_sk_lr_te  = r2_score(y_house_te, sk_lr.predict(X_house_te))
mse_sk_lr_te = mse_loss(y_house_te, sk_lr.predict(X_house_te))

print("GaussianGLM vs. sklearn LinearRegression — California Housing (test set):")
print(f"  Scratch  R^2 = {r2_gauss_te:.4f}  |  MSE = {mse_gauss_te:.4f}")
print(f"  sklearn  R^2 = {r2_sk_lr_te:.4f}  |  MSE = {mse_sk_lr_te:.4f}")
print(f"  Gap: delta_R2 = {abs(r2_gauss_te - r2_sk_lr_te):.4f}")
print("  (Remaining gap due to finite gradient descent steps vs closed-form normal eq.)")

# BernoulliGLM vs. LogisticRegression
sk_logit = LogisticRegression(max_iter=1000, random_state=SEED)
sk_logit.fit(X_iris_tr, y_bin_tr)
acc_sk_logit_te = accuracy_score(y_bin_te, sk_logit.predict(X_iris_te))

print(f"\nBernoulliGLM vs. sklearn LogisticRegression — Iris binary (test acc):")
print(f"  Scratch  acc = {acc_bern_te:.2%}")
print(f"  sklearn  acc = {acc_sk_logit_te:.2%}")

# SoftmaxGLM vs. LogisticRegression (multi_class='multinomial')
sk_soft = LogisticRegression(multi_class="multinomial", solver="lbfgs",
                              max_iter=500, random_state=SEED)
sk_soft.fit(X_iris_tr, y_iris_tr)
acc_sk_soft_te = accuracy_score(y_iris_te, sk_soft.predict(X_iris_te))

print(f"\nSoftmaxGLM vs. sklearn LogisticRegression (multinomial) — Iris 3-class:")
print(f"  Scratch  acc = {acc_soft_te:.2%}")
print(f"  sklearn  acc = {acc_sk_soft_te:.2%}")

In [ ]:
# ── Results Summary Table ─────────────────────────────────────────────────────
summary_data = {
    "GLM": [
        "GaussianGLM (scratch)",
        "sklearn LinearRegression",
        "BernoulliGLM (scratch)",
        "sklearn LogisticRegression",
        "SoftmaxGLM (scratch)",
        "sklearn LogisticRegression (multinomial)",
        "PoissonGLM (scratch)",
    ],
    "Dataset": [
        "California Housing", "California Housing",
        "Iris (binary)", "Iris (binary)",
        "Iris (3-class)", "Iris (3-class)",
        "Synthetic Poisson",
    ],
    "Task": [
        "Regression", "Regression",
        "Classification", "Classification",
        "Classification", "Classification",
        "Count regression",
    ],
    "Test Metric": [
        f"R^2={r2_gauss_te:.4f}",
        f"R^2={r2_sk_lr_te:.4f}",
        f"Acc={acc_bern_te:.2%}",
        f"Acc={acc_sk_logit_te:.2%}",
        f"Acc={acc_soft_te:.2%}",
        f"Acc={acc_sk_soft_te:.2%}",
        f"R^2={r2_pois_te:.4f}",
    ],
}

summary_df = pd.DataFrame(summary_data)
print(summary_df.to_string(index=False))
print()
print("All four GLMs use the identical gradient formula: (1/n) X^T (mu - y).")
print("They differ ONLY in the inverse link function computing mu from eta = X w.")

---
### IRLS — Iteratively Reweighted Least Squares

Gradient descent is not the only way to fit GLMs. The classical algorithm used
by R's `glm()` and most statistical software is **IRLS**: at each step, solve a
*weighted* least-squares problem where the weights are the GLM working weights
`W = diag(mu * (1 - mu))` (for Bernoulli) and the pseudo-response is the
*adjusted dependent variable* `z = eta + (y - mu) / (mu * (1 - mu))`.

IRLS converges in far fewer iterations than gradient descent because it
implicitly uses second-order (Newton) information via the working weights.
It is equivalent to Newton-Raphson on the log-likelihood.

In [ ]:
class BernoulliGLMIRLS:
    """Logistic regression fitted via Iteratively Reweighted Least Squares (IRLS).

    IRLS is the canonical maximum-likelihood algorithm for GLMs. At each step it
    solves a weighted least-squares (WLS) sub-problem using:
        Working weights:       W = diag(mu * (1 - mu))
        Adjusted response:     z = eta + (y - mu) / (mu * (1 - mu))
        WLS normal equations:  w <- (X^T W X)^{-1} X^T W z

    This is mathematically equivalent to Newton-Raphson on the Bernoulli
    log-likelihood. IRLS typically converges in 10-20 iterations vs. hundreds
    for gradient descent.

    Attributes:
        n_iterations: Maximum number of IRLS re-weighting steps.
        tol:          Convergence tolerance on the weight update norm.
        weights_:     Fitted coefficient vector (includes bias), shape (n_features+1,).
        losses_:      BCE loss at each iteration.
        n_iter_:      Actual iterations run before convergence.
    """

    def __init__(
        self,
        n_iterations: int = 25,
        tol: float = 1e-6,
    ) -> None:
        """Initialise IRLS hyperparameters.

        Args:
            n_iterations: Maximum IRLS iterations (typically converges in < 20).
            tol:          Stop early if ||w_new - w_old|| < tol.
        """
        self.n_iterations = n_iterations
        self.tol = tol
        self.weights_: np.ndarray | None = None
        self.losses_: list[float] = []
        self.n_iter_: int = 0

    def fit(self, X_b: np.ndarray, y: np.ndarray) -> "BernoulliGLMIRLS":
        """Fit logistic regression via IRLS (Newton-Raphson on the log-likelihood).

        Args:
            X_b: Design matrix with bias column prepended, shape (n, p+1).
            y:   Binary labels in {0, 1}, shape (n,).

        Returns:
            self, fitted estimator.
        """
        n, p = X_b.shape
        w = np.zeros(p)
        self.losses_ = []

        for iteration in range(self.n_iterations):
            # ── Forward pass ─────────────────────────────────────────────────
            eta = X_b @ w                          # linear predictor (n,)
            mu = sigmoid(eta)                       # predicted probabilities (n,)

            # ── Working weights: W = diag(mu * (1 - mu)) ─────────────────────
            W_diag = mu * (1.0 - mu)               # shape (n,)
            # Clip for numerical stability (avoid division by near-zero)
            W_diag_safe = np.maximum(W_diag, 1e-8)

            # ── Adjusted dependent variable: z = eta + (y - mu) / W_diag ─────
            z = eta + (y - mu) / W_diag_safe       # pseudo-response (n,)

            # ── Weighted Normal equations: w = (X^T W X)^{-1} X^T W z ────────
            # Equivalent to: solve (X^T W X) w = X^T W z
            XtW = X_b.T * W_diag_safe              # (p+1, n)  — broadcasting
            XtWX = XtW @ X_b                        # (p+1, p+1) Fisher info
            XtWz = XtW @ z                          # (p+1,)

            # Add small ridge for numerical stability
            ridge = 1e-8 * np.eye(p)
            w_new = np.linalg.solve(XtWX + ridge, XtWz)

            # ── Record loss ───────────────────────────────────────────────────
            loss = binary_cross_entropy(y, sigmoid(X_b @ w_new))
            self.losses_.append(loss)

            # ── Convergence check ─────────────────────────────────────────────
            delta = np.linalg.norm(w_new - w)
            w = w_new
            self.n_iter_ = iteration + 1
            if delta < self.tol:
                break

        self.weights_ = w
        return self

    def predict_proba(self, X_b: np.ndarray) -> np.ndarray:
        """Return predicted probabilities for the positive class.

        Args:
            X_b: Design matrix with bias column, shape (n, p+1).

        Returns:
            Probability array, shape (n,), values in (0, 1).
        """
        return sigmoid(X_b @ self.weights_)

    def predict(self, X_b: np.ndarray) -> np.ndarray:
        """Return binary class predictions using a 0.5 threshold.

        Args:
            X_b: Design matrix with bias column, shape (n, p+1).

        Returns:
            Integer array of 0/1 predictions, shape (n,).
        """
        return (self.predict_proba(X_b) >= 0.5).astype(int)

    def score(self, X_b: np.ndarray, y: np.ndarray) -> float:
        """Return accuracy on the given design matrix and labels.

        Args:
            X_b: Design matrix with bias column, shape (n, p+1).
            y:   True binary labels, shape (n,).

        Returns:
            Accuracy as a float in [0, 1].
        """
        return float(np.mean(self.predict(X_b) == y))


# ── Compare IRLS vs gradient-descent BernoulliGLM on Iris (binary) ────────────
from sklearn.datasets import load_iris

iris = load_iris()
X_iris_full = iris.data.astype(np.float64)
y_iris_full = iris.target

# Binary task: class 0 vs class 1
mask_bin = y_iris_full < 2
X_bin = X_iris_full[mask_bin]
y_bin = y_iris_full[mask_bin].astype(np.float64)

# Standardise
mu_bin = X_bin.mean(axis=0)
std_bin = X_bin.std(axis=0) + 1e-8
X_bin_std = (X_bin - mu_bin) / std_bin

# Train/test split (80/20)
rng_bin = np.random.default_rng(SEED)
n_bin = len(y_bin)
idx_bin = rng_bin.permutation(n_bin)
split_bin = int(0.8 * n_bin)
tr_idx, te_idx = idx_bin[:split_bin], idx_bin[split_bin:]

X_bin_tr, X_bin_te = X_bin_std[tr_idx], X_bin_std[te_idx]
y_bin_tr, y_bin_te = y_bin[tr_idx], y_bin[te_idx]

# Add bias column
X_bin_tr_b = np.column_stack([np.ones(len(X_bin_tr)), X_bin_tr])
X_bin_te_b = np.column_stack([np.ones(len(X_bin_te)), X_bin_te])

# Fit gradient-descent model
gd_model = BernoulliGLM(learning_rate=0.1, n_iterations=300)
gd_model.fit(X_bin_tr_b, y_bin_tr)

# Fit IRLS model
irls_model = BernoulliGLMIRLS(n_iterations=25, tol=1e-8)
irls_model.fit(X_bin_tr_b, y_bin_tr)

# Report
gd_acc  = gd_model.score(X_bin_te_b, y_bin_te)
irls_acc = irls_model.score(X_bin_te_b, y_bin_te)

print("IRLS vs Gradient Descent (Iris binary)")
print(f"  GD   iterations: {len(gd_model.losses_):>5}  |  test acc: {gd_acc:.4f}")
print(f"  IRLS iterations: {irls_model.n_iter_:>5}  |  test acc: {irls_acc:.4f}")
print(f"  Final GD loss  : {gd_model.losses_[-1]:.6f}")
print(f"  Final IRLS loss: {irls_model.losses_[-1]:.6f}")

# Plot convergence curves
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))

ax1.plot(gd_model.losses_, color="steelblue", linewidth=1.5, label="Gradient Descent (300 steps)")
ax1.set_xlabel("Iteration")
ax1.set_ylabel("BCE Loss")
ax1.set_title("Gradient Descent Convergence")
ax1.legend()
ax1.grid(True, alpha=0.3)

ax2.plot(irls_model.losses_, color="darkorange", linewidth=1.5, marker="o",
         markersize=5, label=f"IRLS ({irls_model.n_iter_} steps)")
ax2.set_xlabel("Iteration")
ax2.set_ylabel("BCE Loss")
ax2.set_title("IRLS Convergence")
ax2.legend()
ax2.grid(True, alpha=0.3)

plt.suptitle("IRLS vs Gradient Descent: Loss Convergence (Iris Binary)", fontsize=12)
plt.tight_layout()
plt.show()

---
### GLM Deviance & Goodness-of-Fit

In classical statistics, GLMs are evaluated via **deviance** rather than loss:

| Quantity | Formula | Meaning |
|---|---|---|
| **Null deviance** | $-2\,\ell(\hat\mu_0; y)$ | Fit of intercept-only model |
| **Residual deviance** | $-2\,\ell(\hat\mu; y)$ | Fit of fitted model |
| **Deviance explained** | $1 - D_{\text{res}} / D_0$ | Analogous to $R^2$ |
| **McFadden's $\rho^2$** | $1 - \ell / \ell_0$ | Log-likelihood ratio |

Lower residual deviance = better fit. The difference $D_0 - D_{\text{res}}$ follows
a $\chi^2$ distribution under the null (deviance test).

In [ ]:
def bernoulli_deviance(y: np.ndarray, mu: np.ndarray) -> float:
    """Compute Bernoulli (binomial) deviance: -2 * log-likelihood.

    The Bernoulli deviance is -2 * sum(y * log(mu) + (1-y) * log(1-mu)),
    which equals 2 * n * BCE_loss. It measures how well the model fits
    relative to a perfect (saturated) model.

    Args:
        y:  True binary labels in {0, 1}, shape (n,).
        mu: Predicted probabilities in (0, 1), shape (n,).

    Returns:
        Scalar deviance value (non-negative float).
    """
    mu_c = np.clip(mu, 1e-10, 1 - 1e-10)
    return float(-2.0 * np.sum(y * np.log(mu_c) + (1.0 - y) * np.log(1.0 - mu_c)))


def poisson_deviance(y: np.ndarray, mu: np.ndarray) -> float:
    """Compute Poisson deviance: 2 * sum(y * log(y/mu) - (y - mu)).

    The Poisson deviance equals 2*(log-likelihood of saturated model -
    log-likelihood of fitted model). Equals zero when mu == y exactly.

    Args:
        y:  True count targets (non-negative integers), shape (n,).
        mu: Predicted means (positive floats), shape (n,).

    Returns:
        Scalar deviance value (non-negative float).
    """
    mu_c = np.maximum(mu, 1e-10)
    # Avoid log(0) for y=0 entries: 0 * log(0) = 0 by convention
    safe_y = np.maximum(y, 1e-10)
    return float(2.0 * np.sum(y * np.log(safe_y / mu_c) - (y - mu_c)))


def gaussian_deviance(y: np.ndarray, mu: np.ndarray) -> float:
    """Compute Gaussian deviance: sum of squared residuals (RSS).

    For Gaussian GLMs the deviance equals the residual sum of squares,
    which is proportional to the MSE loss (deviance = n * MSE * sigma^2).

    Args:
        y:  True continuous targets, shape (n,).
        mu: Predicted means, shape (n,).

    Returns:
        Scalar residual sum of squares (non-negative float).
    """
    return float(np.sum((y - mu) ** 2))


def null_bernoulli_deviance(y: np.ndarray) -> float:
    """Compute null (intercept-only) Bernoulli deviance.

    The null model predicts mu_0 = mean(y) for all observations.

    Args:
        y: True binary labels in {0, 1}, shape (n,).

    Returns:
        Null deviance scalar.
    """
    mu_null = np.full_like(y, y.mean(), dtype=float)
    return bernoulli_deviance(y, mu_null)


def mcfadden_r2(y: np.ndarray, mu: np.ndarray, family: str = "bernoulli") -> float:
    """Compute McFadden's pseudo-R2 for a GLM.

    McFadden's rho^2 = 1 - log_L(fitted) / log_L(null).
    Ranges from 0 (null model) to 1 (perfect fit). Values of 0.2-0.4
    indicate excellent fit in practice (unlike OLS R2 which targets ~1).

    Args:
        y:      True targets, shape (n,).
        mu:     Predicted means, shape (n,).
        family: GLM family -- "bernoulli" or "poisson".

    Returns:
        McFadden pseudo-R2 as a float in [0, 1].

    Raises:
        ValueError: If family is not "bernoulli" or "poisson".
    """
    if family == "bernoulli":
        ll_fitted = -0.5 * bernoulli_deviance(y, mu)
        mu_null = np.full_like(y, y.mean(), dtype=float)
        ll_null = -0.5 * bernoulli_deviance(y, mu_null)
    elif family == "poisson":
        ll_fitted = -0.5 * poisson_deviance(y, mu)
        mu_null = np.full_like(y, y.mean(), dtype=float)
        ll_null = -0.5 * poisson_deviance(y, mu_null)
    else:
        raise ValueError(f"Unsupported family: {family!r}. Use 'bernoulli' or 'poisson'.")
    if ll_null == 0.0:
        return 0.0
    return float(1.0 - ll_fitted / ll_null)


# ── Deviance report for all four GLMs ────────────────────────────────────────
print("=" * 60)
print("GLM Deviance Analysis")
print("=" * 60)

# 1. Gaussian GLM (California Housing)
from sklearn.datasets import fetch_california_housing
housing = fetch_california_housing()
X_h = housing.data.astype(np.float64)
y_h = housing.target.astype(np.float64)
mu_h = X_h.mean(axis=0); std_h = X_h.std(axis=0) + 1e-8
X_h_std = (X_h - mu_h) / std_h
rng_h = np.random.default_rng(SEED)
idx_h = rng_h.permutation(len(y_h))
sp_h = int(0.8 * len(y_h))
tr_h, te_h = idx_h[:sp_h], idx_h[sp_h:]
X_h_tr_b = np.column_stack([np.ones(len(tr_h)), X_h_std[tr_h]])
X_h_te_b = np.column_stack([np.ones(len(te_h)), X_h_std[te_h]])
y_h_tr, y_h_te = y_h[tr_h], y_h[te_h]

g_glm = GaussianGLM(learning_rate=0.05, n_iterations=500)
g_glm.fit(X_h_tr_b, y_h_tr)
mu_g_te = g_glm.predict(X_h_te_b)

d_res_g = gaussian_deviance(y_h_te, mu_g_te)
d_null_g = gaussian_deviance(y_h_te, np.full_like(y_h_te, y_h_tr.mean()))
d_expl_g = 1.0 - d_res_g / d_null_g
print(f"Gaussian GLM   | Null Dev: {d_null_g:10.1f} | Res Dev: {d_res_g:10.1f} | "
      f"Explained: {d_expl_g:.4f}")

# 2. Bernoulli GLM (Iris binary)
mu_b_te = gd_model.predict_proba(X_bin_te_b)
d_res_b = bernoulli_deviance(y_bin_te, mu_b_te)
d_null_b = null_bernoulli_deviance(y_bin_te)
d_expl_b = 1.0 - d_res_b / d_null_b if d_null_b > 0 else 0.0
mf_r2_b = mcfadden_r2(y_bin_te, mu_b_te, family="bernoulli")
print(f"Bernoulli GLM  | Null Dev: {d_null_b:10.4f} | Res Dev: {d_res_b:10.4f} | "
      f"McFadden R2: {mf_r2_b:.4f}")

# 3. Poisson GLM — retrieve from earlier in notebook
# Re-generate synthetic Poisson data and fit
rng_p2 = np.random.default_rng(SEED)
n_p2 = 600
X_p2 = rng_p2.standard_normal((n_p2, 2))
w_true_p2 = np.array([0.8, -0.5, 0.3])  # bias, x1, x2
X_p2_b = np.column_stack([np.ones(n_p2), X_p2])
eta_p2 = X_p2_b @ w_true_p2
y_p2 = rng_p2.poisson(np.exp(np.clip(eta_p2, -5, 5))).astype(float)
sp_p2 = int(0.8 * n_p2)
X_p2_tr_b, X_p2_te_b = X_p2_b[:sp_p2], X_p2_b[sp_p2:]
y_p2_tr, y_p2_te = y_p2[:sp_p2], y_p2[sp_p2:]
p_glm2 = PoissonGLM(learning_rate=0.05, n_iterations=500)
p_glm2.fit(X_p2_tr_b, y_p2_tr)
mu_p2_te = p_glm2.predict(X_p2_te_b)
d_res_p = poisson_deviance(y_p2_te, mu_p2_te)
mu_null_p = np.full_like(y_p2_te, y_p2_tr.mean())
d_null_p = poisson_deviance(y_p2_te, mu_null_p)
d_expl_p = 1.0 - d_res_p / d_null_p if d_null_p > 0 else 0.0
mf_r2_p = mcfadden_r2(y_p2_te, mu_p2_te, family="poisson")
print(f"Poisson GLM    | Null Dev: {d_null_p:10.4f} | Res Dev: {d_res_p:10.4f} | "
      f"McFadden R2: {mf_r2_p:.4f}")

print("=" * 60)
print("Note: Deviance explained = 1 - residual_deviance / null_deviance")
print("      McFadden R2 = 1 - log_L(fitted) / log_L(null)")
print("      Values of 0.2-0.4 indicate excellent GLM fit (not comparable to OLS R2)")

import math


def _regularized_lower_gamma(a: float, x: float, n_terms: int = 150) -> float:
    """Regularized lower incomplete gamma function P(a, x) via series expansion.

    Uses the series P(a, x) = e^{-x} x^a / Gamma(a) * sum_{n>=0} x^n / (a)_{n+1},
    where (a)_{n+1} = a(a+1)...(a+n) is the Pochhammer symbol. Converges
    rapidly for x < a + 1; sufficient for chi-square CDF with small df.

    Args:
        a:       Shape parameter (positive float).
        x:       Upper limit of integration (non-negative float).
        n_terms: Maximum series terms before forced truncation.

    Returns:
        P(a, x) in [0, 1].
    """
    if x <= 0.0:
        return 0.0
    # log-scale accumulation for numerical stability
    log_prefix = -x + a * math.log(x) - math.lgamma(a)
    term = 1.0 / a
    total = term
    for n in range(1, n_terms):
        term *= x / (a + n)
        total += term
        if abs(term) < 1e-14 * abs(total):
            break
    return min(1.0, math.exp(log_prefix) * total)


def chi2_sf(x: float, df: int) -> float:
    """Chi-square survival function (1 - CDF) implemented without scipy.

    The chi-square CDF with df degrees of freedom equals P(df/2, x/2)
    (regularized lower incomplete gamma). The survival function is
    1 - P(df/2, x/2), giving the tail probability for a test statistic x.

    Args:
        x:  Test statistic value (non-negative float).
        df: Degrees of freedom (positive integer).

    Returns:
        P(X > x) where X ~ chi2(df), a float in [0, 1].
    """
    if x <= 0.0:
        return 1.0
    return max(0.0, 1.0 - _regularized_lower_gamma(df / 2.0, x / 2.0))


def deviance_test(
    null_deviance: float,
    residual_deviance: float,
    df_diff: int,
) -> tuple[float, float]:
    """Chi-square deviance test comparing a GLM to the null (intercept-only) model.

    Under H0 (null model), the deviance difference D0 - D_res follows a
    chi-square distribution with df_diff degrees of freedom (= number of
    predictors added). A small p-value means the predictors collectively
    improve fit significantly.

    Args:
        null_deviance:     Deviance of the intercept-only model.
        residual_deviance: Deviance of the fitted model.
        df_diff:           Degrees of freedom = number of added predictors.

    Returns:
        Tuple (chi2_stat, p_value).
            chi2_stat: Deviance difference (test statistic).
            p_value:   P-value from chi-square survival function (no scipy).
    """
    chi2_stat = float(max(0.0, null_deviance - residual_deviance))
    p_value = chi2_sf(chi2_stat, df=df_diff)
    return chi2_stat, p_value


# Deviance test for the Bernoulli GLM (4 predictors added over null)
chi2_b, pval_b = deviance_test(d_null_b, d_res_b, df_diff=4)
chi2_p, pval_p = deviance_test(d_null_p, d_res_p, df_diff=2)

print("\nChi-square Deviance Tests")
print(f"  Bernoulli GLM : chi2 = {chi2_b:.4f},  p = {pval_b:.4e}  (df=4)")
print(f"  Poisson GLM   : chi2 = {chi2_p:.4f},  p = {pval_p:.4e}  (df=2)")
print("  (p < 0.05 means predictors significantly improve fit over null model)")

---
## Part 4 — Evaluation & Analysis

### The Unified Gradient — Proving the GLM Insight

The GLM's central claim is that ALL four models share the same gradient formula.
We demonstrate this empirically by computing and comparing gradient norms
across all four GLMs at random parameter initializations.

In [ ]:
# ── Demonstrate unified gradient formula across all GLMs ─────────────────────

def verify_glm_gradient(
    X_b: np.ndarray,
    y: np.ndarray,
    w: np.ndarray,
    link_fn: object,
    loss_fn: object,
    epsilon: float = 1e-5,
) -> tuple[np.ndarray, np.ndarray]:
    """Numerically verify the GLM analytical gradient against finite differences.

    Args:
        X_b:     Feature matrix with bias, shape (n, p).
        y:       Targets, shape (n,).
        w:       Weight vector, shape (p,).
        link_fn: Inverse link function: eta -> mu.
        loss_fn: Loss function: (y, mu) -> scalar.
        epsilon: Finite difference step size.

    Returns:
        Tuple (analytical_grad, numerical_grad) — both shape (p,).
    """
    mu = link_fn(X_b @ w)
    analytical_grad = glm_gradient(X_b, y, mu)

    numerical_grad = np.zeros_like(w)
    for j in range(len(w)):
        w_plus = w.copy()
        w_minus = w.copy()
        w_plus[j]  += epsilon
        w_minus[j] -= epsilon
        loss_plus  = loss_fn(y, link_fn(X_b @ w_plus))
        loss_minus = loss_fn(y, link_fn(X_b @ w_minus))
        numerical_grad[j] = (loss_plus - loss_minus) / (2.0 * epsilon)

    return analytical_grad, numerical_grad


rng_verify = np.random.default_rng(SEED + 42)
n_verify, d_verify = 80, 3
X_verify_raw = rng_verify.standard_normal((n_verify, d_verify))
X_verify_b   = np.column_stack([np.ones(n_verify), X_verify_raw])
w_rand       = rng_verify.normal(0, 0.5, d_verify + 1)

# Gaussian
y_verify_gauss = X_verify_b @ w_rand + rng_verify.normal(0, 0.5, n_verify)
a_gauss, n_gauss = verify_glm_gradient(
    X_verify_b, y_verify_gauss, w_rand,
    lambda eta: eta,  # identity link
    mse_loss,
)
err_gauss = np.max(np.abs(a_gauss - n_gauss))

# Bernoulli
p_verify = sigmoid(X_verify_b @ w_rand)
y_verify_bern = rng_verify.binomial(1, p_verify)
a_bern, n_bern = verify_glm_gradient(
    X_verify_b, y_verify_bern.astype(float), w_rand,
    sigmoid, binary_cross_entropy,
)
err_bern = np.max(np.abs(a_bern - n_bern))

# Poisson
y_verify_pois = rng_verify.poisson(np.exp(np.clip(X_verify_b @ w_rand, -5, 5)))
a_pois, n_pois = verify_glm_gradient(
    X_verify_b, y_verify_pois.astype(float), w_rand,
    poisson_inverse_link, poisson_nll,
)
err_pois = np.max(np.abs(a_pois - n_pois))

print("Gradient Verification (analytical vs numerical finite-difference):")
print(f"  GaussianGLM   max |grad error|: {err_gauss:.2e}  (should be ~1e-7)")
print(f"  BernoulliGLM  max |grad error|: {err_bern:.2e}  (should be ~1e-7)")
print(f"  PoissonGLM    max |grad error|: {err_pois:.2e}  (should be ~1e-7)")
print()
print("All three use the SAME formula glm_gradient(X, y, mu) = (1/n) X^T (mu - y).")
print("This is the mathematical elegance of GLMs: one gradient to rule them all.")

In [ ]:
# ── Loss Curve Visualisation for All Four GLMs ───────────────────────────────
fig, axes = plt.subplots(1, 4, figsize=(16, 4))

glm_infos = [
    (glm_gauss, "GaussianGLM\n(MSE)", "steelblue",  "California Housing",  "MSE"),
    (glm_bern,  "BernoulliGLM\n(BCE)", "darkorange", "Iris binary",         "Binary cross-entropy"),
    (glm_soft,  "SoftmaxGLM\n(CCE)",  "firebrick",  "Iris 3-class",        "Categorical cross-entropy"),
    (glm_pois,  "PoissonGLM\n(NLL)",  "purple",     "Synthetic Poisson",   "Poisson NLL"),
]

for ax, (glm, name, color, dataset, ylabel) in zip(axes, glm_infos):
    steps = [i * 20 for i in range(len(glm.losses_))]
    ax.plot(steps, glm.losses_, color=color, linewidth=2)
    ax.set_xlabel("Iteration")
    ax.set_ylabel(ylabel, fontsize=9)
    ax.set_title(f"{name}\n{dataset}", fontsize=10)

plt.suptitle("GLM Training Convergence — All Four Models", fontsize=12)
plt.tight_layout()
plt.show()

print("All four GLMs converge smoothly. The identical gradient formula means")
print("that the same gradient descent code works across all models.")

In [ ]:
# ── Coefficient Interpretation: Connecting GLM Weights to the Data ───────────
fig, axes = plt.subplots(1, 3, figsize=(15, 4))

# GaussianGLM: bar chart of feature coefficients (skip bias term)
coefs_gauss = glm_gauss.weights_[1:]  # skip bias
axes[0].barh(feature_names_house, coefs_gauss, color="steelblue", alpha=0.8)
axes[0].axvline(0, color="k", linewidth=0.8)
axes[0].set_title("GaussianGLM Coefficients\n(California Housing)", fontsize=10)
axes[0].set_xlabel("Coefficient value")

# SoftmaxGLM: weight matrix as heatmap (skip bias row)
W_soft = glm_soft.weights_[1:, :]  # shape (4, 3)
im = axes[1].imshow(W_soft.T, cmap="coolwarm", aspect="auto")
axes[1].set_xticks(range(4))
axes[1].set_xticklabels([n[:8] for n in feature_names_iris], rotation=30, fontsize=8)
axes[1].set_yticks(range(3))
axes[1].set_yticklabels(class_names_iris)
axes[1].set_title("SoftmaxGLM Weight Matrix\n(Iris 3-class)", fontsize=10)
plt.colorbar(im, ax=axes[1], shrink=0.8)

# PoissonGLM: predicted vs actual scatter
y_pois_pred_tr = glm_pois.predict(X_pois_tr_b)
axes[2].scatter(y_pois_tr, y_pois_pred_tr, alpha=0.4, s=15, color="purple")
lim = max(y_pois_tr.max(), y_pois_pred_tr.max()) + 1
axes[2].plot([0, lim], [0, lim], "r--", linewidth=1.5, label="Perfect")
axes[2].set_xlabel("Observed count y")
axes[2].set_ylabel("Predicted rate mu = exp(Xw)")
axes[2].set_title(f"PoissonGLM: Predicted vs Actual\nR^2={r2_pois_tr:.4f}", fontsize=10)
axes[2].legend(fontsize=9)

plt.suptitle("GLM Coefficient Analysis", fontsize=12)
plt.tight_layout()
plt.show()

In [ ]:
# ── Error Analysis: Softmax GLM on Iris (3-class) ────────────────────────────
y_pred_soft_te = glm_soft.predict(X_iris_te_b)

fig, axes = plt.subplots(1, 2, figsize=(11, 4))

# Confusion matrix
cm = confusion_matrix(y_iris_te, y_pred_soft_te)
ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=class_names_iris).plot(
    ax=axes[0], colorbar=False)
axes[0].set_title(f"SoftmaxGLM Confusion Matrix\nIris test acc={acc_soft_te:.2%}",
                  fontsize=11)

# Predicted probability distributions per class
proba_te = glm_soft.predict_proba(X_iris_te_b)
for k, class_name in enumerate(class_names_iris):
    axes[1].hist(proba_te[:, k], bins=15, alpha=0.5, label=class_name, density=True)
axes[1].set_xlabel("Predicted probability")
axes[1].set_ylabel("Density")
axes[1].set_title("Predicted Probability Distributions\n(Iris test set)", fontsize=11)
axes[1].legend(fontsize=9)

plt.tight_layout()
plt.show()

# Classification report
print("SoftmaxGLM — Classification Report (Iris 3-class, test set):")
print(classification_report(y_iris_te, y_pred_soft_te,
                             target_names=class_names_iris))

# Error cases
error_mask = y_pred_soft_te != y_iris_te
n_errors = int(error_mask.sum())
print(f"Misclassified samples: {n_errors} / {len(y_iris_te)}")
if n_errors > 0:
    print("Misclassification details:")
    for idx in np.where(error_mask)[0]:
        true_cls  = class_names_iris[y_iris_te[idx]]
        pred_cls  = class_names_iris[y_pred_soft_te[idx]]
        max_prob  = proba_te[idx].max()
        print(f"  True={true_cls:10s}  Pred={pred_cls:10s}  "
              f"Max prob={max_prob:.3f}")

---
## Part 5 — Summary & Lessons Learned

### Key Takeaways

1. **One framework, four models:** Gaussian, Bernoulli, Poisson, and Categorical
   distributions are all members of the exponential family
   $p(y;\eta) = b(y)\exp(\eta T(y) - a(\eta))$. Each yields a GLM by choosing
   the canonical link $g(\mu) = \eta = \mathbf{w}^\top \mathbf{x}$.

2. **Universal gradient formula:** For any canonical GLM, the gradient of the
   negative log-likelihood is always $\nabla_\mathbf{w}\mathcal{L} = \frac{1}{n}\mathbf{X}^\top(\boldsymbol{\mu} - \mathbf{y})$.
   The four models we built share a single `glm_gradient()` function — the only
   difference is the inverse link computing $\boldsymbol{\mu}$.

3. **Why these loss functions?** The cross-entropy loss in logistic regression
   and the MSE in linear regression are not arbitrary choices — they are the
   negative log-likelihoods of the Bernoulli and Gaussian distributions respectively.
   The GLM framework makes this derivation systematic.

4. **Sufficient statistics compress information:** The natural parameter $\eta$
   and the log-partition $a(\eta)$ encode all information about the distribution.
   The mean $\mu = a'(\eta)$ and variance $a''(\eta)$ are derivatives of $a$ —
   a powerful result that gives free moment formulas.

5. **Forward to deep learning:** The softmax output layer in a neural network
   (Module 5-04) is exactly the SoftmaxGLM applied as the final layer. The link
   function determines the output activation: sigmoid for binary, softmax for
   multi-class, linear for regression. GLM theory explains *why* these activation
   functions are paired with specific loss functions.

### What's Next

→ **02-07 (k-Nearest Neighbors)** introduces the polar opposite of GLMs — a
lazy, non-parametric model that makes no distributional assumptions and defers
all computation to prediction time.

### Going Further

- **Iteratively Reweighted Least Squares (IRLS):** The Newton-Raphson analogue
  for GLMs converges in fewer steps than gradient descent by using the Fisher
  information matrix as the Hessian — see *Nelder & Wedderburn (1972)*.
- **Regularised GLMs:** Ridge (L2) and Lasso (L1) penalties extend naturally to
  all four GLMs; sklearn's `LogisticRegression(C=...)` implements L2 BernoulliGLM.
- **Mixed models:** When observations are not independent (e.g., repeated
  measures), Generalised Linear *Mixed* Models (GLMMs) add random effects.
- **Dispersion parameter:** The Gaussian GLM above fixes $\sigma^2 = 1$;
  in practice the dispersion parameter is estimated from residuals.